# 14 — Claims, conclusions and the paper outline (System I, N3)

The old notebook 14 synthesised the notebooks 02–12 of the pre-fix pipeline. This one is **rebuilt from the claims ledger** of `docs/Manuscript/system_i_rerun_plan.md` (copied in at build time, sections 1 and 3) and from a **dashboard that loads every headline number from the stored results** (section 2; nothing is typed here).
Section 4 lists what is not settled. Section 5 (the conclusions) is written after the last run.


## 1. The storyline and the SI decisions (from the plan)

### Working storyline of the paper (agreed with the user 2026-11-09)

**Core message.** Simulation-based inference gives calibrated, near-optimal posteriors for fault
diagnosis in closed-loop processes, where the controller masks faults that open-loop methods
misread. But the posterior inherits every assumption of the simulator: the control law, the
operating history before the observation window (warm start), the sensor-noise model and the
inlet conditions can each produce overconfidence, bias or spurious α–β correlations, and these
failures are *silent*: the posterior looks confident and plausible. SBI is reliable if the
simulator tells the truth, and easy to get wrong otherwise.

**Structure.**
1. Motivating application, Claim 1: closed-loop SBI diagnoses faults the controller masks;
   open-loop-trained SBI fails, visibly, with no posterior mass inside the prior (nb07).
2. Core contribution, sources of silent failure, each with its experiment:
   control law (controller bug; nb30 counterfactual) · warm start (nb29: P0 reproduces the old
   β bias, P1 gives β overconfidence and a spurious correlation whose sign depends on the sensor
   model; P2/P1b side by side) · sensor model (nb33: N0–N4 ladder, artefact correlations, α
   without C) · inlet conditions (R2, notebook 34: done, not robust, silent; see the ledger) · **operating-mode mismatch** (nb07: a closed-loop posterior applied to open-loop data is biased and overconfident, β̂ +0.026 and 90% coverage 0.36, with only 0.08% of its samples outside the prior, so the prior-leakage diagnostic does **not** flag it: a silent failure; the converse mismatch, an open-loop posterior on closed-loop data, is flagged: 100% outside the prior).
3. Practical take-away, sensor specification: fouling (β) is diagnosable from T, Tc, Qc alone;
   catalyst activity (α) needs an online concentration measurement (nb33 §7).
4. Practitioner checklist (closing section), one item per experiment: real control law ·
   warm start matching the plant's history · sensor model of the real instruments, with the
   measurements each fault needs · inlet uncertainty as nuisance parameters · operating mode of the plant matches the simulator (prior leakage catches a gross mismatch, not a mild one) · joint calibration
   checks (SBC + TARP) · precision against the information limit (Cramér–Rao bounds).
   **Added 2026-11-09 (nb11b, nb09):** sensor-fault guard (alarm) plus the drift-aware posterior [α, β, δT] as the estimate under an alarm (nb09: Sc7 β coverage 0.90, bias −0.024; price: β sd ×1.2–1.9, macro-F1 −0.03) · · z_T = (mean measured T − T_sp)/(σ_T/√n) on the controlled variable, applied only to unsaturated windows (threshold calibrated on healthy simulated windows, ≈ 10 here; Sc7 flagged 100%, false positives 0.9%); a saturated valve is reported as loss of control.

**Production model (decided 2026-11-09 after nb35).** The paper's production posterior is conditioned on
measured inlet temperatures (ε = 0.1 K, notebook 36). The nominal-inlet posterior (nb31) is the
*naive* model: it shows the silent failure under realistic inlet errors (nb34) and is the model of
the nominal-inlet Claim 1 comparison (nb07). Headline precision and accuracy come from notebook 36.

**Evidence still needed for the "reliable" half:** Claim 1 under N3 (nb07), baselines
(NUTS, EKF/UKF), and R2 (β under a 0.5–1 K feed-temperature error, with and without Ti as a
nuisance parameter). Whatever R2 shows goes in: robustness, or another checklist item.

**For commentary cells.** Tie each result to one of the four parts above; say explicitly
whether it supports the "reliable" half (calibrated, near the bound, correct diagnosis) or
documents a silent failure mode (and how it would be detected). Report weaker numbers under
realistic sensors as findings explained by the sensor ladder, not as setbacks.



### SI material (user decisions 2026-11-09)

### SI-1 Why there is no NUTS baseline (decision: option C)
The earlier NUTS baseline (`cstr_generative_model`: deterministic diffrax summary + full-covariance Gaussian on the 29 summary statistics) is invalid under the N3 + P2 model, and that explains the old R-hat of about 500 (nb23)
without either of the two gradient hypotheses recorded in HANDOFF.md. Read-only probe `scripts/nuts_diag_likelihood_mean.py` (`results/nuts_diag_likelihood_mean.csv`; 400 replicates of the D0 generator per closed-loop scenario against the deterministic summary):
* the warm start is the **exact steady state**, so the deterministic trajectory is flat; the observed windows are not (process noise, N3 sensor noise, the P2 history). The deterministic mean is **5–16 replicate sd away** on 10–13 of the 29 features in every scenario (T_std 0 against 1.56 K, z = −15.7; int_abs_T_err 0 against 75, z = −14.5; Tc_std −8.3; T_min/T_max +6.3/−6.4; Tc_min +5.3; C_std −5.7);
* the 29-D replicate **covariance is numerically singular** (condition numbers 4e16 to 6e301): some features are exact linear functions of others. A full-covariance Gaussian with a mean off by 5–16 sd along near-null directions yields gradients dominated by the mismatch;
* the older runs with a clean R-hat used an under-converged warm start whose transient dynamics mimicked the missing variance, which is why they looked fine before nb22's exact steady state.
Consequence: no NUTS number from that likelihood is usable. A valid exact-likelihood baseline does not exist for this stochastic closed-loop simulator; the alternatives considered were a mean-feature Gaussian likelihood and a synthetic likelihood with an emulator, and were not built (approximate, large effort). The posterior is instead checked by SBC, TARP, the Cramér–Rao bound (nb31), the EKF/UKF comparison (nb16) and the individual-posterior test L-C2ST (nb37). Notebook 21 (open-loop NUTS ground truth) is SI only, with this diagnosis; NUTS is not rebuilt.

### SI-2 L-C2ST: the compressed-observation variant is an open item, and the sbi default is not reproducible on raw inputs (user decision 2026-11-10: record, no further work)
Notebook 37 (individual-posterior test of the production ensemble, 25 windows from D1):
* **Reproducibility finding.** The sbi default (`LC2ST`, raw inputs, sklearn MLP with (20, 20) hidden units) gave three different answers in three runs with identical seeds and data: 13 of 25 windows below p = 0.05 (median p 0.04), then 0 of 25 (median 0.10), then 0 of 25 (median 0.62). The cause is most likely the unscaled inputs (features from 0.02 mol/L to 400 L/min), which make the classifier training numerically unstable; it was not isolated. The first run's 13 of 25 is an artefact. Do not use the sbi default on unscaled inputs for this problem; standardise first.
* **Headline.** The standardised-input variant (power demonstrated: it rejects β shifted by +0.10, 2.3 posterior sd, and a halved width in 25 of 25 windows) rejects 0 of 25 windows of the production posterior (minimum p 0.19, median 0.54).
* **Open item (not pursued).** The variant with the observation compressed to the ensemble's own posterior mean and sd (6 classifier inputs) flags 8 of 25 windows of the production posterior (Sc1 2, Sc2 2, Sc3 1, Sc4 3, Sc5 0; median p 0.08, KS p = 0.000 against uniform) and 25 of 25 of the wrong posterior. It is not known whether this is a local miscalibration as a function of the posterior's own mean and sd (invisible to the global SBC and the coverage tables) or an uncalibrated null of that variant; the full-input test with the same information finds nothing. Resolving it would need more windows and a second calibration set. Recorded here as an open item for the SI; no further work.



## 2. The headline numbers, loaded from the stored results

In [1]:
import json, os, sys
from pathlib import Path
import numpy as np, pandas as pd
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "system_I" and NB_DIR.parent.name == "notebooks", f"run from notebooks/system_I (cwd {NB_DIR})"
ROOT = NB_DIR.parent.parent; R = ROOT / "results"
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
def need(*names):
    miss = [n for n in names if not (R / n).exists()]
    assert not miss, f"missing result files: {miss}"
need("36_N3_production.json", "36_N3_metrics.csv", "11_N3_macro_f1.csv", "11_N3_L1_three_numbers.csv", "10_N3_tracking_metrics.csv", "16_N3_tuned_vs_untuned_pooled_Sc1_5.csv", "09_N3_decision.json",
     "38_N3_gate_rates.csv", "08_N3_fim.csv", "18_N3_ablation.csv", "11b_N3_zT_guarded_rule.csv")
rows = []
j = json.loads((R / "36_N3_production.json").read_text()); m36 = pd.read_csv(R / "36_N3_metrics.csv"); e = m36[(m36.member == "ensemble") & (m36.budget == j["production_budget"])].iloc[0]
rows.append(("Production posterior (nb36)", f"{j['production_budget']} simulations x {len(j['members'])}-member ensemble; features {j['summary_set']}; interior SBC KS p alpha {e.ks_p_a_interior:.3f}, beta {e.ks_p_b_interior:.3f}; TARP p {e.tarp_ks_p:.3f}; 90% coverage {e.cov90_a:.3f} / {e.cov90_b:.3f}; sd {e.sd_a:.3f} / {e.sd_b:.3f}"))
mf = pd.read_csv(R / "11_N3_macro_f1.csv"); r = mf[(mf.set.str.startswith("L1")) & (mf.scenarios.str.startswith("Sc1-Sc5"))].iloc[0]; r2 = mf[(mf.set.str.startswith("L1")) & (mf.scenarios == "all six")].iloc[0]
rows.append(("Macro-F1 (nb11, L1 = 6,000 windows)", f"Sc1-Sc5 {r.macro_f1:.3f} (accuracy {r.accuracy:.3f}); with Sc7 {r2.macro_f1:.3f}"))
t = pd.read_csv(R / "11_N3_L1_three_numbers.csv"); rows.append(("Per-scenario accuracy / beta coverage / beta sd (nb11, L1)", "; ".join(f"{a.scenario} {a.class_acc:.3f} / {a.cov90_b:.3f} / {a.sd_b:.3f}" for a in t.itertuples())))
tr = pd.read_csv(R / "10_N3_tracking_metrics.csv").iloc[0]; rows.append(("30-day tracking (nb10)", f"alpha MAE {tr.alpha_MAE:.3f}, beta MAE {tr.beta_MAE:.3f}; 90% coverage {tr.alpha_cov90:.3f} / {tr.beta_cov90:.3f}; beta bias {tr.beta_bias:+.3f} (persistent inlet offsets); class accuracy {tr.class_acc:.3f}"))
pooled = pd.read_csv(R / "16_N3_tuned_vs_untuned_pooled_Sc1_5.csv")
rows.append(("EKF/UKF vs SBI (nb16, D1 Sc1-Sc5 pooled)", "; ".join(f"{a.method} {a.variant}: beta coverage {a.cov90_b:.2f}, RMSE {a.rmse_b:.3f}, tail windows {int(a.n_tail_b)}" for a in pooled.itertuples()) + f"; for comparison SBI production (nb11, L1 = 1,000 windows per scenario, a different set): mean beta coverage {t[t.scenario != 'Sc7'].cov90_b.mean():.2f}, mean sd {t[t.scenario != 'Sc7'].sd_b.mean():.3f}"))
dec = json.loads((R / "09_N3_decision.json").read_text()); rows.append(("Drift-aware posterior (nb09)", f"verdict {dec['verdict']}; macro-F1 Sc1-Sc5 {dec['macro_f1_Sc1_5']:.4f}; Sc7 beta coverage {dec['sc7']['cov90_b']:.3f}, bias {dec['sc7']['bias_b']:+.3f}"))
g = pd.read_csv(R / "38_N3_gate_rates.csv"); rows.append(("Gate (nb38, D1)", f"false-alarm rate (all windows) {', '.join(f'{a.scenario} {a.flag_rate_all_windows:.2f}' for a in g[g.scenario != 'Sc7'].itertuples())}; Sc7 detection {g[g.scenario == 'Sc7'].flag_rate_all_windows.iloc[0]:.2f} overall, {g[g.scenario == 'Sc7'].flag_rate_unsaturated.iloc[0]:.2f} unsaturated"))
f = pd.read_csv(R / "08_N3_fim.csv"); n_ = f[f["mode"].str.startswith("nominal")]; u_ = f[f["mode"].str.startswith("nuisance")]
rows.append(("Information (nb08)", f"CRB beta nominal {n_.crb_sd_b.min():.3f}-{n_.crb_sd_b.max():.3f}; inlet as a nuisance {u_.crb_sd_b.min():.3f}-{u_.crb_sd_b.max():.3f}; alpha {u_.crb_sd_a.min():.3f}-{u_.crb_sd_a.max():.3f} (unchanged)"))
ab = pd.read_csv(R / "18_N3_ablation.csv").set_index("set"); rows.append(("Feature ablation (nb18)", "; ".join(f"{k.split(' (')[0]}: beta sd {v.sd_b:.3f}, macro-F1 {v.macro_f1:.2f}" for k, v in ab.iloc[[0, 1, 3, 4, 5]].iterrows())))
for nm, fn, fmt in (("Sbi timing (nb12)", "12_N3_sbi_timing.csv", None), ("Break-even (nb12)", "12_N3_break_even.csv", None)):
    if (R / fn).exists():
        d_ = pd.read_csv(R / fn); rows.append((nm, d_.round(3).to_string(index=False)))
for fn, nm in (("37_N3_lc2st_positive_controls.csv", "L-C2ST positive controls (nb37)"),):
    if (R / fn).exists():
        d_ = pd.read_csv(R / fn); rows.append((nm, "; ".join(f"{a.variant}|{a.posterior}: p<0.05 in {a.frac_p_below_0_05:.2f}" for a in d_.itertuples())))
if (R / "39_N3_decision.json").exists():
    j39 = json.loads((R / "39_N3_decision.json").read_text()); rows.append(("Saturated-window drift detector, plug-in beta disagreement (nb39)", f"tau {j39['tau']:.2f}; D1 saturated Sc7 detection {j39['detection_Sc7_saturated']:.3f}, Sc5 false alarms {j39['false_alarm_Sc5_saturated']:.3f}; pre-set bar met: {j39['bar_met']}"))
dash = pd.DataFrame(rows, columns=["claim", "value (loaded from results/)"])
for _, r_ in dash.iterrows():
    print(f"* {r_['claim']}\n    {r_.iloc[1]}\n")
dash.to_csv(R / "14_N3_dashboard.csv", index=False)

* Production posterior (nb36)
    20000 simulations x 5-member ensemble; features S1m_measured_inlet; interior SBC KS p alpha 0.063, beta 0.030; TARP p 0.710; 90% coverage 0.919 / 0.905; sd 0.036 / 0.043

* Macro-F1 (nb11, L1 = 6,000 windows)
    Sc1-Sc5 0.907 (accuracy 0.912); with Sc7 0.824

* Per-scenario accuracy / beta coverage / beta sd (nb11, L1)
    Sc1 0.970 / 0.916 / 0.044; Sc2 0.857 / 0.904 / 0.044; Sc3 0.992 / 0.925 / 0.045; Sc4 0.766 / 0.910 / 0.045; Sc5 0.977 / 0.927 / 0.043; Sc7 0.341 / 0.002 / 0.029

* 30-day tracking (nb10)
    alpha MAE 0.040, beta MAE 0.038; 90% coverage 0.885 / 0.878; beta bias -0.029 (persistent inlet offsets); class accuracy 0.675

* EKF/UKF vs SBI (nb16, D1 Sc1-Sc5 pooled)
    EKF untuned: beta coverage 0.68, RMSE 0.093, tail windows 8; UKF untuned: beta coverage 0.74, RMSE 0.075, tail windows 2; EKF tuned: beta coverage 0.76, RMSE 0.087, tail windows 6; UKF tuned: beta coverage 0.67, RMSE 0.044, tail windows 1; for comparison SBI production (nb1

## 3. The claims ledger

### Claims ledger (as of the build of this notebook)

| Manuscript claim | Section | Produced by | Status |
|---|---|---|---|
| Claim 1: closed-loop SBI diagnoses faults the controller masks; an open-loop-trained model fails (visibly: no posterior mass inside the prior) | §6.1 (motivating application) | nb07 (N3) | **done (N3 + P2 + S1, 40k ensemble), 2026-11-09:** Sc2 (β = 0.85): CL-SBI β̂ 0.848, coverage 0.94, accuracy 0.98, W1 0.0153 [N0 0.861, 0.94, 0.96, 0.0285]; OL-SBI has **100%** of its samples outside the prior (N0 94%), W1 0.304 [0.437] — **fails, here visibly** (the N0 wording "reads the fouled plant as healthy", accuracy 0.28, is retired: N3 raw β̂ 0.77, accuracy 0.64 on out-of-support samples, not quotable; what is robust is the out-of-support failure). Sc6: reversal holds for W1 (0.0108 vs 0.0275) and coverage (0.88 vs 0.36), not for class accuracy (0.98 both; N0 0.88 vs 0.38). Leakage flags the catastrophic mismatch, not the mild one (CL on open-loop data: 0.08% outside, coverage 0.36 = silent). Single OL seed, 50 windows |
| SBC KS p (α, β) of the production posterior | §6.1 | nb31 (N3) | **done (N3 + P2 + S1), 2026-11-09:** **deployed = 40k, 5-seed ENSEMBLE** (rule v2 found convergence at 20k→40k; v1 would give 10k; N0: 160k ensemble). Calibrated at every budget 5k–160k: interior KS α 0.22–0.88, β 0.04–0.81 (40k: 0.879 / 0.080), TARP KS 0.45–0.76 (40k 0.762), 90% coverage 0.90–0.91. **Amendment v2.1 (2026-11-09, after results, user-approved):** the deployed posterior must itself pass calibration (d); v2 would have deployed seed 3 (interior β KS p = 0.00005; 2 of 5 members fail on β, all pass TARP) on a tie (TARP dev 0.0200 vs 0.0200, below the ~0.043 sampling noise); v2 choice kept in the record; N0 unchanged. [N0 160k ensemble: interior α 0.024, β 0.84, TARP 0.86 → SI] |
| Sensor-model ladder N0–N4 (which measurement buys which diagnosis) | SI / §6.3 | nb33 | done: N3 α sd 0.03–0.06, β 0.015, no α–β confounding; N4 α weakly identified (sd/prior sd 0.67–0.91, not ≈ 1 — corrected 2026-11-09), β fine, macro-F1 0.69 ± 0.02 (prior draws) / 0.53 ± 0.05 (scenarios) |
| Classification F1 / confusion per sensor model; artefact α–β correlations (Sc5 under imprecise T, N4 without C, Sc7 drift); α without C | §6.3 / SI figure | nb33 §7 | **done (N3), 2026-11-09:** macro-F1 N3 0.929 ± 0.006 (prior draws) / 0.900 ± 0.020 (scenarios) [N0 0.967 ± 0.002 / 0.987 ± 0.007]; errors in the α-only class pairs; Sc4 0.88; **Sc7 (drift) 0.46 under N3 (N0 1.00, N2 0.00)** — drift robustness open → nb09. Sc5 ridge N0 +0.67 ± 0.04 → N3 +0.10 ± 0.03; N4 creates correlations (+0.15 to +0.51, Fisher +0.37 to +0.78); fouling class recall 0.96 → 0.49 without C while β is unaffected |
| Drift-aware posterior [α, β, δT] (remedy for the Sc7 silent failure) | §6.x / checklist | nb09 (N3) | **done, 2026-11-09, pre-set verdict FIXED (thin margins):** Sc7 β bias −0.395 → −0.024, β coverage 0.002 → 0.900, class accuracy 0.341 → 0.820, δT̂ recovered (coverage 0.92, sd 0.2 K); price: β sd 0.054–0.084 vs 0.044, macro-F1 Sc1–Sc5 0.907 → 0.877 (limit 0.877, margin 0.0001), TARP p 0.017, single seeds fail SBC (ensemble required); covers an additive T-sensor offset within ±3 K only |
| Deployment combination: z_T gate + drift-aware posterior (D1) | §6.x / checklist | nb38 | **done, 2026-11-09:** guarded gate (unsaturated, \|z_T\| > 10): false alarms 0–2% per scenario, Sc7 detection 1.00 unsaturated / 0.62 overall; Sc1–Sc5 = production (macro-F1 0.939); **Sc7 roughly halved, not removed** (coverage 0.00 → 0.52, bias −0.394 → −0.153): the 19 saturated Sc7 windows stay wrong (coverage 0.00). Without the guard Sc7 is repaired (coverage 0.88) at false alarms 10–76% and macro-F1 −0.045 |
| Individual-posterior check (replaces the NUTS ground truth): L-C2ST of the production ensemble | §6.1 / SI | nb37 | **done, 2026-11-09:** standardised-input L-C2ST (power demonstrated: rejects β shifted by +0.10 and a halved width in 25 of 25 windows) finds **0 of 25 windows** (5 per scenario Sc1–Sc5, D1) below p = 0.05 (min 0.19, median 0.54); the sbi default on raw inputs is not reproducible (13/25, 0/25, 0/25 across identical-seed runs): not used; compressed-x variant flags 8 of 25 (open question). Not independent of the simulator: checks the model, not the plant |
| Resource analysis: SBI vs EKF/UKF cost and break-even | §6.4 / SI | nb12 (N3) | **done, 2026-11-09:** SBI 51 ms per single window, 13 ms per window batched; EKF 32 ms; UKF 484 ms; training 6.0 min (retraining again 6.0 min); break-even vs UKF 766–833 windows, vs EKF 18,700 (batched) or never (single-window); NUTS not compared (decision C) |
| Drift detector in saturated windows (plug-in β disagreement d) | §6.x / checklist | nb39 | **done, 2026-11-10, pre-set bar NOT met:** threshold fixed on a separate saturated-Sc5 set (τ = 1.05; AUC 0.96 there); on D1 saturated windows Sc7 detection **0.105 (2 of 19)**, Sc5 false alarms 0.054, Sc1–Sc4 0.000; the saturation caveat of nb38 stands (neither z_T nor d detects a +2 K offset reliably when the valve is saturated); extended gate not evaluated |
| Posterior precision vs Cramér–Rao bound | §6.1 / SI | nb31 (N3) | **done (N3, vs the production set's own bound S1, nb03 10b):** posterior sd / bound at 40k: α 0.98–1.02, β 0.85–1.04 (S0 bound: β 0.77–0.95); 160k similar; 20k 4–10% above for α. Raw-trajectory bound NOT a valid β reference under N3 (Ledoit–Wolf looser: 0.031 vs 0.014–0.016). Absolute: α sd 0.027–0.064, β 0.014–0.024 [N0: α 0.002–0.003, β 0.027–0.028 at Sc4: α 15× less precise, β 2× more precise under realistic sensors]. Diagnosis (40k ens): **Sc4 accuracy 0.886 ± 0.010 on 1,000 fresh windows = the production figure to quote**; the fixed 50-window test set gives 0.76–0.80 over budgets (quote only as the fixed-test-set figure: those windows are harder than average, binomial SE 0.057, shared by all budgets; not a P2 or S1 effect: nb29 E3 0.889 vs 0.900, S0 vs S1 on the same windows 0.846 vs 0.849); Sc7 (drift) 0.32 with β̂ 0.47: a silent failure from unmodelled sensor drift |
| Macro-F1 over closed-loop scenarios | abstract, §6.2 | nb05a / nb11 | **done (nb11, production posterior, L1 = 1,000 windows per scenario), 2026-11-09:** **0.907** over Sc1–Sc5 (accuracy 0.912; D1 0.936 on 250 windows); 0.824 with Sc7. Per-class F1: healthy 0.911, fouling 0.926, decay 0.946, combined 0.845. Old 0.927 invalid |
| Per-scenario α̂, β̂, 90% CI, coverage (Table 7) | §6.2 | nb11 | **done (nb11 L1), 2026-11-09:** accuracy / β coverage / β sd: Sc1 0.970 / 0.916 / 0.044; Sc2 0.857 / 0.904 / 0.044; Sc3 0.992 / 0.925 / 0.045; Sc4 0.766 / 0.910 / 0.045; Sc5 0.977 / 0.927 / 0.043; **Sc7 0.341 / 0.002 / 0.029 (β bias −0.395: silent failure)**. Calibrated in the saturated and unsaturated windows separately; saturated windows wider (β sd 0.045–0.058 vs 0.036–0.042). Sc4 0.766 here vs 0.886 in nb31 (nominal inlet): the ±2 K inlet costs accuracy, not calibration. Naive posterior on D1: coverage 0.02–0.08, accuracy 0.24–0.58 (sensitivity section only) |
| I_αα / I_ββ ratio, full covariance | §6.3.1 | nb29 E2 → nb08 | **done (nb08 N3, 2026-11-09), the old 380–1500 is NOT reproduced:** I_ββ/I_αα = 12.5 / 10.4 / 2.9 / 6.2 / 7.9 (Sc1–Sc5) with the inlet nominal (I_αα/I_ββ 0.08–0.35), **1.4 / 1.6 / 0.35 / 0.78 / 1.5 with the inlet as a nuisance** (β information falls ×4.5–9, α unchanged within ~15%); CRB β 0.014–0.023 nominal → 0.043–0.048 nuisance; production posterior sd / CRB(nuisance): β 0.91–1.07, α 1.02–1.16; implied corr(α, β) +0.04–0.13 nominal, +0.14–0.24 nuisance (max Sc5); saturation costs β ×1.5 (nominal inlet), nothing under the nuisance |
| ∂T_ss/∂β = 0 argument | §6.3.2 | analytic + nb01 | valid within a window now (10–26 min settling) |
| CNN vs hand-crafted comparison | §6.3.3 | nb04b/24 | pending; "within 1%" claim retired |
| Baselines table (SBI, NUTS, EKF, UKF) | §6.4 | nb05/23, nb16 | **EKF/UKF done (nb16 N3, D1 snapshot, 50 windows/scenario), 2026-11-09; NUTS and 30-day tracking pending:** **fair tuning (c3, 2026-11-09) does not close the β calibration gap:** EKF β coverage 0.68→0.76, UKF 0.74→0.67 (tuned UKF RMSE 0.044 = SBI sd but reported sd 0.025); the filter tail windows are one failure (cold feed, valve at the lower limit; SBI fine there); aug filters (measured inlet + offset states) have typical β error ≈ SBI (RMSE 0.045 excluding rare outliers; SBI sd 0.044) but are overconfident (β coverage 0.58–0.86 vs SBI 0.88–0.96) with unflagged outlier windows (EKF 8/250, UKF 2/250 with \|β err\|>0.15); without offset states coverage 0.30–0.48; naive filters collapse on D1 (RMSE 0.30–0.45, coverage ≤0.08) like the naive posterior; Sc7 UKF-aug β bias −0.38, coverage 0.00 (same failure as SBI); EKF 31 ms, UKF 360–480 ms per window |
| 30-day tracking MAE | §6.5 | nb10 (N3) | **done, 2026-11-09 (new stream built in `ws_phase3.generate_stream`; `scenarios.generate_degradation_stream` left unchanged and unused):** α MAE 0.040, β MAE 0.038; 90% coverage α 0.885 / β 0.878; α bias +0.005, **β month-mean bias −0.029 ± 0.001 from persistent Ti/Tci offsets (+0.02, +0.155 K)**; class accuracy 0.675 (threshold-limited); detection delay β 61 h, α 201 h; z_T detector flags 99% of saturated and 11% of unsaturated windows (needs a saturation guard); SBI 53 ms per single window, 9.8 ms batched |
| Initial-state protocol effect (old β bias explained) | new section | nb29 | **done (N3), 2026-11-09 (P2 test column, 3 seeds, 5k sims):** abrupt-fault (P0) and noise-free (P1) training fail under both sensor models (TARP dev 0.271 / 0.091 vs 0.041–0.059 for P1b/P2/P3; β coverage 0.41 / 0.72 vs 0.86–0.87); the noise-free start still overstates β precision (Fisher sd 1.8× / 3.3× too small at Sc4, summaries / raw) and invents an α–β correlation whose **sign flips with the sensor model** (Fisher at Sc4: N0 −0.69 / −0.50, N3 +0.36 / +0.52; SBI N0 −0.11, N3 +0.38 ± 0.05). Rule selects **P2** over P1b by a small margin (one P1b seed fails the 0.01 α-KS gate; RMSE 0.0402 for both); user adopted P2, P1b reported side by side. [N0: P0→P2 old Sc2 (β = 0.70) bias −0.145 vs −0.149 manuscript; N0 rule chose P1b, flagged.] E4 (window length) not run (RUN_E4 off). The E2 "Sc2" point is β = 0.70 (saturating), not today's Sc2 |
| Hand-crafted summaries close to sufficient | §6.3.3 | nb29 E2 | done (N0): raw-trajectory CRB within ~10–20% of the summary CRB. **N3 (2026-11-09):** α raw 0.045 vs summaries 0.051 (Sc1, P1b: raw ~12% tighter); for β the Ledoit–Wolf raw bound is *looser* (0.031 vs 0.016), a shrinkage artefact — do not cite β "summaries exceed raw"; use the summary bound as the β reference |
| Summary-statistic redesign (kept or replaced, and why) | §4.2 / SI | nb03 §10 | done: 29-D kept. Candidate features add no information (CRB equal within 1%); β already at its CRB; α 3–5× wider than its CRB (a learning gap, closed by budget: 5k→20k halves α sd); plug-in estimators narrow α ~30% but make it overconfident (interior KS fails, 90% cov 0.88). S1 kept as documented fallback. **N3 + P2 (2026-11-09):** the pre-set rule selects **S1_base_est** (29 + 3 plug-in estimators) by clause (b): S0 TARP KS p 0.0056 (gate 0.01), S1 0.048; RMSE 0.0370 vs 0.0385; efficiency tied (0.867 ± 0.010 vs 0.880 ± 0.014); S1's β bound 7–10% tighter, α identical. Marginal (3 seeds): user adopted S1 as the N3 production set, S0 shown side by side; N0 stays S0. Production features: `ws_eval.features`, `system_I_config.summary_set()`; R2 (notebook 34) must test S1 with S0 as the comparison (the estimators use the nominal inlet) |
| α–β correlation ≈ 0 in control, positive (heat-balance compensation) under valve saturation | discussion | nb29 → nb32 | **done (nb32 N3, 2026-11-09), in a weaker form than assumed:** positive and small everywhere (production posterior on D1 +0.10–0.15; naive on D0 +0.04–0.09; FIM-implied +0.04–0.13 nominal, +0.14–0.24 nuisance), larger in saturated windows (+0.16–0.24 against +0.07–0.13 unsaturated); no banana; unmeasured Ci alone costs β 10–30% in the bound, no systematic effect on the correlation; the mechanism is not isolated |
| k0 = 60× Fogler, ~98% conversion | §3.1 | nb25 | done (disclosure text pending) |
| Controller-form correction | §3.1 / SI | nb30 scripts | done (text pending) |
| Warm-start protocol under realistic sensors (P2 chosen; P1b side by side; P0/P1 fail under both sensor models) | new section | nb29 (N3) | decided 2026-11-09: P2 production; interpretation pending |
| Robustness to inlet uncertainty (Ti, Tci, Ci); misspecification alarm from the two β estimates | §6.x / checklist | nb34 (R2) | **done (N3 + P2 + S1, deployed 40k ensemble), 2026-11-09: NOT robust; a silent failure.** Pre-set rule fails: 18 of 24 gated cells violate it (β coverage ≥ 0.80 in 6 of 24; only Ci −1% passes). Pooled Sc1–Sc4 β bias (own sd; coverage; accuracy): Ti ∓0.5 K +0.021 (1.2; 0.67; 0.95) / −0.048 (2.4; 0.30; 0.96); **Tci −0.5 K −0.144 (7.4; 0.00; 0.52), +0.5 K +0.216 (13.0; 0.00; 0.50)**; Ci ∓1% +0.008 (0.6; 0.84; 0.96) / −0.015 (0.9; 0.72; 0.97); β sd stays 0.015–0.020 (confident and wrong). The coolant inlet Tci dominates (jacket-balance estimate ±14–46% per 0.5 K; Tc − Tci is a few K); the β = 0.85 scenarios are pushed to the jacket limit by +0.5 K on the feed (Sc2 59%, Sc4 56% of samples at Qc ≥ 395) but saturation amplifies and does not cause it (Ti −0.5 K, 0% saturated, coverage 0.62–0.72; Tci biases 0–1% saturated). S1 is not more fragile than S0 (Tci −0.5 K −0.143 vs −0.140; Tci +0.5 K +0.205 vs +0.148; Ti, Ci less). Alarm (|z| > 3, false alarm 0.00): detects Tci ±0.5 K in 0.40–0.52 (0.80–0.90 for fouled Sc2/Sc4, ≤ 0.16 for healthy/decay), Ti ±0.5 K 0.02–0.03, Ci ≈ 0 — partial. Part B (reported, not adopted): robust training (Ti, Tci ~ N(297, 0.5²), Ci 1%) restores calibration under the realistic inlet distribution (β KS 0.76, TARP 0.81, coverage 0.888 vs 0.161 for the nominal posterior) at ~5× the β sd (0.015 → 0.079) and class accuracy 0.97 → 0.82. Checklist: measure or marginalise Tci (≈ 0.04 K), Ti (≈ 0.15 K), Ci (≈ 1%) |
| Measured inlet temperatures as inputs: instrument requirement (ε) and precision cost vs marginalising | §6.x / checklist | nb35 (R2 Part C) | **done (N3 + P2), 2026-11-09: the nb34 silent failure is removed** (pooled β coverage 0.90–0.95, mean \|bias\| < 0.02 at all 13 inlet conditions within ±2 K, against production 0.00–0.47 and bias 0.035–0.31). β sd at the nominal inlet: production 0.015 (nominal inlet assumed), oracle 0.017, **ε 0.05 K 0.028, ε 0.1 K 0.039, ε 0.2 K 0.055**, marginal 0.111; accuracy 0.94 / 0.93 / 0.905 / 0.82 / 0.655. **Instrument requirement the data support: coolant (and feed) inlet temperature to ≤ 0.1 K.** **Pre-set rule: no variant deployable → mechanically recommends "marginal", which is the worst variant (29 violations, accuracy 0.655); ε 0.05 and ε 0.1 each violate one of 52 cells (coverage 0.78 vs 0.80; accuracy 0.86 vs oracle 1.00 at the threshold-near Sc2); the oracle itself fails TARP (KS p 0.0004).** Decision for the user (not adopted): follow the rule, or propose ε = 0.1 K / 0.05 K, or amend the fallback |
| Production model with measured inlet (ε = 0.1 K): budget, calibration, precision, robustness over ±2 K | §6.1 (headline) | nb36 | **done (N3 + P2), 2026-11-09:** **20k, 5-seed ensemble** (rule v2.1, unchanged, not flagged; only 2 of 5 single members pass calibration); ensemble calibrated (interior KS α 0.063, β 0.030, TARP KS 0.71; coverage 0.919 / 0.905); β sd 0.044 / 0.043, α sd 0.056 / 0.044 (Sc1 / Sc4); over ±2 K: coverage 0.84–0.94 (mean 0.937), \|bias\| 0.005, accuracy 0.907, against the naive nominal-inlet posterior 0.00 / 0.166 / 0.610. **Acceptance missed by the pre-set count (4 of 52 cells, class accuracy only, Sc2/Sc4 at β = 0.85); accepted by the user with a documented exception, criterion not amended:** a real accuracy cost of ε = 0.1 K near the class threshold (1,000-window follow-up: Sc2 0.857–0.921 vs oracle 0.985–0.994, ε 0.05 K 0.935–0.976; Sc4 0.803 vs 0.877, 0.846; `results/36_N3_acceptance_review.json`). Trade-off: ε 0.1 K β sd 0.04; ε 0.05 K 0.03; oracle 0.017. **Open:** downstream notebooks need the measured Ti, Tci per window (nb02 data has no inlet measurement, to be checked); Claim 1 stays a nominal-inlet comparison |


## 4. What is not settled (read before the conclusions)

* **No NUTS baseline** (decision C, plan SI-1): the posterior is checked by SBC, TARP, the Cramér–Rao bound, the EKF/UKF comparison and L-C2ST. The notebooks 05, 20, 21 and 23 keep their pre-fix NUTS numbers under a banner that they are not valid; **notebook 05a** (pre-fix NUTS classification) also uses `run_mcmc_baseline` and has no banner yet.
* **Drift-aware posterior (nb09):** pre-set verdict FIXED with a macro-F1 margin of 0.0001 and an ensemble TARP p of 0.017; the deployment gate (nb38) repairs the unsaturated Sc7 windows only. **Saturated windows (nb39):** the plug-in β disagreement does not detect the drift there either (pre-set bar not met: Sc7 detection 0.105 on D1, false alarms 0.054), so the saturation caveat stands: neither z_T nor d works with a saturated valve; routing every saturated window to the drift-aware posterior (the no-guard variant of nb38) costs macro-F1 and width.
* **The L-C2ST (nb37):** the standardised variant (power demonstrated) rejects 0 of 25 windows; the sbi default on raw inputs is not reproducible (not used); the compressed-observation variant flags 8 of 25 windows and is an open item (plan SI-2).
* **System II / Luyben** carry the same malformed PI form as System I had; not touched.
* **CNN vs hand-crafted (nb04b/24)** is still pending; the old "within 1%" claim is retired.
* **The k0 = 60× Fogler** (about 98% conversion) is kept and must be disclosed.
* **Controller-form correction, k0 disclosure and the warm-start protocol** are decided but their manuscript text is pending (ledger).
* **SI figures:** regenerated from stored data and results only by the new notebook 17 (`17_supplementary_figures_N3.ipynb`) into `figures/si_N3/` (data/ verified unchanged by checksums); the SI still cites the old files in `figures/` until the swap is decided. The old notebook 17 rewrites `data/observations.npz` and must not be run.


## 5. Conclusions

*Written 2026-11-09 against the dashboard of section 2 and the ledger of section 3; every number is in one of them.*

**Core message.** Simulation-based inference gives calibrated, near-optimal posteriors for fault diagnosis in a closed-loop process (the controller masks faults that open-loop methods misread), **provided the simulator tells the truth**. The production posterior (20,000 simulations × 5 seeds, measured inlet temperatures as inputs) is calibrated (90% coverage 0.92 for α and 0.91 for β on the SBC set, TARP p 0.71), unbiased in the well-specified scenarios
(|bias| ≤ 0.016 for α and ≤ 0.007 for β on D1), and **sits at the information bound of its summaries** (posterior sd / Cramér–Rao bound 0.91–1.07 for β and 1.02–1.16 for α once the inlet is treated as a nuisance). It reaches a macro-F1 of **0.907** over the closed-loop scenarios Sc1–Sc5. A posterior with an individual-window test of power (L-C2ST, standardised) finds no error in 25 windows.
It is **easy to get wrong otherwise**: the control law, the warm start, the sensor model, the inlet conditions, the operating mode and a sensor drift can each produce overconfidence, bias or a spurious (α, β) correlation, and the posterior looks equally confident and plausible when they do.

**The sources of silent failure, each with its experiment.**
* *Control law:* the PI law lacked the proportional gain on the integral term (fixed 2026-11-08; every earlier cache was invalid; k0 kept at 60× Fogler and disclosed).
* *Warm start (nb29):* abrupt-fault and noise-free starts (P0, P1) fail under both sensor models (TARP deviation 0.27 and 0.09 against 0.04–0.06 for the noisy histories; β coverage 0.41 and 0.72 against 0.86–0.87); the noise-free start overstates the precision of β (Fisher sd 1.8–3.3× too small) and invents an α–β correlation whose sign flips with the sensor model. The production start is P2 (a 300-minute noisy history).
* *Sensor model (nb33):* the N0–N4 ladder shows what each measurement buys: β is diagnosable from T, Tc and Qc alone; α needs an online concentration measurement; a coarse concentration sensor creates artefact correlations.
* *Inlet conditions (nb34 → nb35 → nb36):* a posterior trained for a nominal inlet is **not robust and fails silently** (β coverage 0.00 under a 0.5 K coolant-inlet error with its sd unchanged at 0.015–0.020; 0.00–0.47 across ±2 K; it also collapses on the paper's own D1 data: coverage 0.02–0.08, and its prior leakage rises to 38%). Measuring the inlet temperatures removes the failure (coverage 0.84–0.94, accuracy 0.907 across ±2 K) at a price of a factor of 3 in the β sd (0.014 → 0.043;
  the Cramér–Rao view, notebook 08), and the instrument requirement the data support is **coolant and feed inlet temperature to ≤ 0.1 K**. The feature ablation (nb18) confirms it: without any inlet information the β sd is 0.126; the measured temperatures alone restore 0.043.
* *Operating mode (nb07):* a closed-loop posterior applied to open-loop data is biased and overconfident; prior leakage flags a gross mismatch (100% of the samples outside the prior) but not a mild one (0.08%, coverage 0.36).
* *Sensor drift (nb09, nb11, nb11b, nb38):* a +2 K drift of the temperature sensor leaves the posterior confidently wrong (β biased by −0.395, coverage 0.002). It is **not silent in the data**: a residual on the controlled variable (z_T) flags it in every unsaturated window at a false-alarm rate of 0.9% (threshold ≈ 10, saturation guard). Making the offset a parameter of the posterior remedies the failure when that posterior is used for every window (Sc7 coverage 0.90, bias −0.024)
  at a price (β sd ×1.2–1.9, macro-F1 −0.03; the pre-set verdict FIXED has thin margins). The deployment combination (gate → drift-aware posterior) repairs the unsaturated Sc7 windows and leaves the saturated ones wrong; neither the temperature residual nor the disagreement of the two plug-in β estimators detects the drift when the valve is saturated (nb39: detection 0.105 on D1), so the treatment of saturated windows is a decision, not a result.
* *Persistent offsets over time (nb10):* over 30 days of drifting inlet the posterior tracks α and β (MAE 0.040 and 0.038; coverage 0.89 / 0.88), but a persistent inlet calibration offset costs a **level bias** (β −0.029 at +0.155 K): the trend is robust, the level, and so any alarm threshold on it, is not.

**The classical comparison (nb16, nb12).** An EKF/UKF with the same model, sensors and measured inlet (and offset states) matches the posterior's typical β accuracy (RMSE 0.044–0.045 after removing rare failures) but is **overconfident** (β coverage 0.67–0.76 after a fair tuning, against 0.91–0.93) and fails in one operating region without a warning (cold feed, valve at the lower limit; the posterior is fine there). The naive filters collapse on the varying inlet exactly as the naive posterior does, so the inlet failure is a property of the model, not of SBI.
Cost: 51 ms per window for SBI (13 ms batched), 32 ms for an EKF, 484 ms for a UKF; training 6 minutes; break-even against the UKF at about 800 windows, against the EKF at about 18,700 (batched) or never (single window).

**Sensor-specification message.** Fouling (β) is diagnosable from T, Tc and Qc, but the coolant inlet temperature must be known to 0.1 K; catalyst activity (α) needs an online concentration measurement; a controlled-variable sensor needs a residual check.

**Practitioner checklist (one item per experiment).** Real control law · warm start matching the plant's history · the sensor model of the real instruments and the measurements each fault needs · inlet uncertainty measured to ≤ 0.1 K or treated as nuisance parameters · operating mode of the plant matches the simulator (prior leakage catches a gross mismatch, not a mild one) · joint calibration checks (SBC and TARP, ensemble of seeds) · precision against the information limit (Cramér–Rao) · an individual-posterior test (L-C2ST with demonstrated power) · **a sensor-fault guard** (z_T on the controlled variable, unsaturated windows only, threshold calibrated on healthy simulated windows, saturated valve reported as loss of control) with the drift-aware posterior as the estimate under an alarm.

**What the paper should not claim.** An NUTS ground truth (there is none; plan SI-1); near-perfect classification near the class threshold (Sc2, Sc4: accuracy 0.86 and 0.77 on L1); a robust posterior for any sensor fault other than a constant offset of the temperature sensor within ±3 K; a negative (α, β) correlation (it is small and positive); that the posterior is exact (the individual-posterior test and the information bound check consistency with the model, not with the plant).

**Open items.** Section 4.